# Orientation: Mahony Filter using Quaternions

The complementary filter employed earlier was insufficient for the project needs. Therefore, here the Mahony filter is tested out to try to remedy its shortcomings.

## Frames

The world frame has to be defined more precisely for downstream computations. An orthonormal base in $\mathbb{R}^3$ can be defined based on the convention outlined in the previous notebook. When the athlete starts recording, he should generally stand upright for a few seconds. This will define a fixed reference for the world frame with its origin at the athlete's initial position, with respect to which the orientation of the body frame will be measured. A process that resembles Gram-Schmidt orthonormalization (theorem 6.1.2 from [1]) can be used to obtain the basis vectors. Let $\hat{\bm{x}}$, $\hat{\bm{y}}$, and $\hat{\bm{z}}$ denote the unit vectors along the x, y, and z axes, respectively. When the device is stationary, since the z-axis is aligned with the direction of specific force, it follows: 
$$
\boxed{\hat{\bm{z}}^b_w = \frac{\bm{\bar{f}}^b}{\|\bm{\bar{f}}^b\|}}
$$
Where $\hat{\bm{z}}^b_w$ is the world frame z-axis expressed in the body frame and $\bm{\bar{f}}^b$ is the mean over the initialization window of measured specific force in the body frame.

For the y-axis, theorem 4.2.5 from [1] about vector projection states that for a vector $\bm{v} \ne \bm{0}$, a vector $\bm{w}$ is orthogonal to $\bm{v}$ when it satisfies the following formula:
$$
\bm{w} = \bm{u} - \frac{\bm{u} \cdot \bm{v}}{\|\bm{v}\|^2} \bm{v}
$$
For any other vector $\bm{u} \ne \bm{0}$. Let $\bm{v} = \hat{\bm{z}}^b_w$, $\bm{w} = \bm{y}^b_w$ and $\bm{u} = \hat{\bm{y}}^b = (0, 1, 0)^T$, and it follows that:
$$
\boxed{\bm{y}^b_w = \hat{\bm{y}}^b - (\hat{\bm{y}}^b \cdot \hat{\bm{z}}^b_w)\hat{\bm{z}}^b_w = \hat{\bm{y}}^b - proj_{\hat{\bm{z}}^b_w} \hat{\bm{y}}^b}
$$
$$
\boxed{\hat{\bm{y}}^b_w = \frac{\bm{y}^b_w}{\|\bm{y}^b_w\|}}
$$
Note that these formulas break when $\hat{\bm{y}}^b$ is parallel/antiparallel to $\hat{\bm{z}}^b_w$, as the projection would result in a zero vector. This can be seen clearly in the following picture:

<img src="./frames/proy.png" alt="Vector projection" width="350" height="200">

This situation happens when the the athlete begins recording while being horizontal, and can be checked with $||\bm{y}^b_w|| < \text{tol}$. In this case, choose $\bm{u} = \hat{\bm{z}}^b = (0, 0, 1)^T$ and repeat the computation. The case in which the athlete begins recording while being sideways is already covered with the first choice of $\bm{u}$, as it's linearly independent with $\hat{\bm{z}}^b_w$.

The $\hat{\bm{x}}$-axis can be obtained by taking the cross product of the y and z axes, being careful of preserving right-handedness:
$$
\boxed{\bm{x}^b_w = \hat{\bm{y}}^b_w \times \hat{\bm{z}}^b_w}
$$
$$
\boxed{\hat{\bm{x}}^b_w = \frac{\bm{x}^b_w}{\|\bm{x}^b_w\|}}
$$
These vectors can be used to assemble a coordinate frame transformation matrix from the body frame to the world frame:
$$
\bm{C}^w_b = (\bm{C}^b_w)^{-1} = (\bm{C}^b_w)^T = \begin{bmatrix}
\hat{\bm{x}}^b_w & \hat{\bm{y}}^b_w & \hat{\bm{z}}^b_w
\end{bmatrix}^T
$$
This holds since a matrix formed by orthonormal column vectors is orthogonal, meaning its inverse is equal to its transpose. (theorem 6.1.3 from [1]). Finally:
$$
\boxed{\bm{C}^w_b = \begin{bmatrix}
\hat{\bm{x}}^b_w & \hat{\bm{y}}^b_w & \hat{\bm{z}}^b_w
\end{bmatrix}^T}
$$
This matrix can be later be transformed into a quaternion representation for use in the orientation estimation algorithm.

### Sources:

1. [Álgebra lineal - Stanley I. Grossman](../../docs/misc/Álgebra%20lineal%20-%20Stanley%20Grossman.pdf) - Chapters 4, 5 and 6.

## Rotations in 3D space: Quaternions

### Overview

Quaternions are an extension of the complex numbers that live in the four-dimensional space. They are useful because they can be used to represent rotations in 3D space, while, at the same time, avoiding the gimbal lock problem associated with Euler angles. They can be represented as:
$$
\bm{q} = (q_0, q_1, q_2, q_3) = q_0 + q_1 \bm{i} + q_2 \bm{j} + q_3 \bm{k} = (s, \bm{v})
$$
where, in the last notation, $s = q_0$ is the "scalar part" and $\bm{v} = ( q_1, q_2, q_3 )$ is the "vector part". 

### Basic operations

Quaternions support addition, subtraction, multiplication, and division, similar to complex numbers. Addition and subtraction work on the components individually. For multiplication, the associative property as well as the following identities can be used:
$$
\bm{i}^2 = \bm{j}^2 = \bm{k}^2 = -1 \\
\bm{i} \bm{j} = - \bm{j} \bm{i} = \bm{k} \\
\bm{j} \bm{k} = - \bm{k} \bm{j} = \bm{i} \\
\bm{k} \bm{i} = - \bm{i} \bm{k} = \bm{j} \\
$$
Note inmediately from these identities that the product is not commutative. Let $\bm{p} = (w, \bm{u})$ be another quaternion. A formula in matrix form for the product can be derived:
$$
\bm{q} \otimes \bm{p} =
\begin{bmatrix}
q_0 & -q_1 & -q_2 & -q_3 \\
q_1 & q_0 & -q_3 & q_2 \\
q_2 & q_3 & q_0 & -q_1 \\
q_3 & -q_2 & q_1 & q_0
\end{bmatrix}
\begin{bmatrix}
p_0 \\
p_1 \\
p_2 \\
p_3
\end{bmatrix} =
\begin{bmatrix}
s w - \bm{v} \cdot \bm{u} \\
s \bm{u} + w \bm{v} + \bm{v} \times \bm{u} \\
\end{bmatrix} 
\ne \bm{p} \otimes \bm{q}
$$

### Norm, conjugate, and inverse

The norm of a quaternion $\bm{q}$ is simply the Euclidean norm of its components:
$$
\|\bm{q}\| = \sqrt{q_0^2 + q_1^2 + q_2^2 + q_3^2}
$$
It can be used to normalize a quaternion:
$$
\hat{\bm{q}} = \frac{\bm{q}}{\|\bm{q}\|}
$$

The conjugate of a quaternion $\bm{q}$ is obtained by negating its vector part:
$$
\bm{q}^* = (q_0, -q_1, -q_2, -q_3)
$$
The conjugate of the product of two quaternions is:
$$
(\bm{q} \otimes \bm{p})^* = \bm{p}^* \otimes \bm{q}^*
$$
The inverse of a quaternion $\bm{q}$ is given by:
$$
\bm{q}^{-1} = \frac{\bm{q}^*}{\|\bm{q}\|^2}
$$
This can be employed to perform division of quaternions, being aware that of the order of multiplication matters, and that writing $\frac{\bm{q}}{\bm{p}}$ is ambiguous.

### Rotations

To rotate a vector $\bm{u} \in \mathbb{R}^3$ using a unit quaternion $\hat{\bm{q}}$, first form a pure quaternion $\bm{p} = (0, \bm{u})$, then apply the rotation as follows:
$$
\bm{p}' = (0, \bm{u}') = \hat{\bm{q}} \otimes \bm{p} \otimes \hat{\bm{q}}^*
$$
Here $\hat{\bm{q}}$ can be expressed as:
$$
\hat{\bm{q}} = \cos(\frac{\beta}{2}) + \sin(\frac{\beta}{2}) (n_1 \bm{i} + n_2 \bm{j} + n_3 \bm{k})
$$
Using this notation, $\hat{\bm{n}} = (n_1, n_2, n_3) \in \mathbb{R}^3$ represents the axis of rotation, and rotation is performed by an angle $\beta$ around this axis, or equivalently, on the plane perpendicular to this axis. This can easily be seen interactively in the source [2]. 

Alternatively, using vector algebra relations [5] and half-angle trigonometric formulas [6], the quaternion that rotates a vector $\bm{u}$ to the direction of a vector $\bm{u'}$ using the shortest arc possible is:
$$
\hat{\bm{q}}
= (\cos(\frac{\beta}{2}), \sin(\frac{\beta}{2}) \hat{\bm{n}})
= \left(\sqrt{\frac{1 + \cos(\beta)}{2}}, \sqrt{\frac{1 - \cos(\beta)}{2}} \hat{\bm{n}} \right)
= \left(\sqrt{\frac{1 + \hat{\bm{u}} \cdot \hat{\bm{u'}}}{2}}, \sqrt{\frac{1 - \hat{\bm{u}} \cdot \hat{\bm{u'}}}{2}} \frac{\hat{\bm{u}} \times \hat{\bm{u'}}}{\|{\hat{\bm{u}} \times \hat{\bm{u'}}}\|} \right)
$$
Note that this formula assumes that $\bm{u}$ and $\bm{u'}$ are not collinear (i.e. $\bm{u} \times \bm{u'} \neq \bm{0}$). If they are collinear, the rotation axis is not uniquely defined. If they are parallel (i.e. $\bm{u} = k \bm{u'}$, where $k \in \mathbb{R}^+$, or equivalently $\hat{\bm{u}} = \hat{\bm{u'}}$), use $\hat{\bm{n}} = \bm{0}$, since no rotation is needed. If they are antiparallel (i.e. $\bm{u} = -k \bm{u'}$, or equivalently $\hat{\bm{u}} = -\hat{\bm{u'}}$), a $180^\circ$ rotation is needed, so use the basis vector least aligned with $\bm{u}$ as $\hat{\bm{n}}$.

### Conversion to other representations

Quaternions are hard visualize on a time series plot directly, but they can be converted to Euler angles with the following formulas [3]:
$$
\phi_{bw} = atan2(2(q_{b,0}^w q_{b,1}^w + q_{b,2}^w q_{b,3}^w), 1 - 2((q_{b,1}^w)^2 + (q_{b,2}^w)^2)) \\
\theta_{bw} = asin(2(q_{b,0}^w q_{b,2}^w - q_{b,1}^w q_{b,3}^w)) \\
\psi_{bw} = atan2(2(q_{b,0}^w q_{b,3}^w + q_{b,1}^w q_{b,2}^w), 1 - 2((q_{b,2}^w)^2 + (q_{b,3}^w)^2))
$$
To go the opposite way, use:
$$
q_{b,0}^w = \cos\left(\frac{\phi_{bw}}{2}\right) \cos\left(\frac{\theta_{bw}}{2}\right) \cos\left(\frac{\psi_{bw}}{2}\right) + \sin\left(\frac{\phi_{bw}}{2}\right) \sin\left(\frac{\theta_{bw}}{2}\right) \sin\left(\frac{\psi_{bw}}{2}\right) \\
q_{b,1}^w = \sin\left(\frac{\phi_{bw}}{2}\right) \cos\left(\frac{\theta_{bw}}{2}\right) \cos\left(\frac{\psi_{bw}}{2}\right) - \cos\left(\frac{\phi_{bw}}{2}\right) \sin\left(\frac{\theta_{bw}}{2}\right) \sin\left(\frac{\psi_{bw}}{2}\right) \\
q_{b,2}^w = \cos\left(\frac{\phi_{bw}}{2}\right) \sin\left(\frac{\theta_{bw}}{2}\right) \cos\left(\frac{\psi_{bw}}{2}\right) + \sin\left(\frac{\phi_{bw}}{2}\right) \cos\left(\frac{\theta_{bw}}{2}\right) \sin\left(\frac{\psi_{bw}}{2}\right) \\
q_{b,3}^w = \cos\left(\frac{\phi_{bw}}{2}\right) \cos\left(\frac{\theta_{bw}}{2}\right) \sin\left(\frac{\psi_{bw}}{2}\right) - \sin\left(\frac{\phi_{bw}}{2}\right) \sin\left(\frac{\theta_{bw}}{2}\right) \cos\left(\frac{\psi_{bw}}{2}\right)
$$
Being aware that these formulas reconstruct the active rotation: $R_b^w = R_z(\psi)R_y(\theta)R_x(\phi)$.

Quaternions can also be converted to coordinate transformation matrices using the following formula [3]:
$$
\bm{C}^w_b = \begin{bmatrix}
(q_{b,0}^w)^2 + (q_{b,1}^w)^2 - (q_{b,2}^w)^2 - (q_{b,3}^w)^2 & 2(q_{b,1}^w q_{b,2}^w + q_{b,3}^w q_{b,0}^w) & 2(q_{b,1}^w q_{b,3}^w - q_{b,0}^w q_{b,2}^w) \\
2(q_{b,1}^w q_{b,2}^w - q_{b,0}^w q_{b,3}^w) & (q_{b,0}^w)^2 - (q_{b,1}^w)^2 + (q_{b,2}^w)^2 - (q_{b,3}^w)^2 & 2(q_{b,2}^w q_{b,3}^w + q_{b,0}^w q_{b,1}^w) \\
2(q_{b,1}^w q_{b,3}^w + q_{b,0}^w q_{b,2}^w) & 2(q_{b,2}^w q_{b,3}^w - q_{b,0}^w q_{b,1}^w) & (q_{b,0}^w)^2 - (q_{b,1}^w)^2 - (q_{b,2}^w)^2 + (q_{b,3}^w)^2
\end{bmatrix}
$$

To go the opposite way, first calculate:
$$
d_0 = 1+C_{b,11}^w+C_{b,22}^w+C_{b,33}^w \\
d_1 = 1+C_{b,11}^w-C_{b,22}^w-C_{b,33}^w \\
d_2 = 1-C_{b,11}^w+C_{b,22}^w-C_{b,33}^w \\
d_3 = 1-C_{b,11}^w-C_{b,22}^w+C_{b,33}^w \\
$$
Then, determine which $d_k$ is largest and use the corresponding formula to compute the quaternion components.

If $d_0$ is largest:
$$
q_{b,0}^w=\frac{\sqrt{\max(d_0,0)}}{2},\quad
q_{b,1}^w=\frac{C_{b,32}^w-C_{b,23}^w}{4q_{b,0}^w},\quad
q_{b,2}^w=\frac{C_{b,13}^w-C_{b,31}^w}{4q_{b,0}^w},\quad
q_{b,3}^w=\frac{C_{b,21}^w-C_{b,12}^w}{4q_{b,0}^w}.
$$
If $d_1$ is largest:
$$
q_{b,1}^w=\frac{\sqrt{\max(d_1,0)}}{2},\quad
q_{b,0}^w=\frac{C_{b,32}^w-C_{b,23}^w}{4q_{b,1}^w},\quad
q_{b,2}^w=\frac{C_{b,12}^w+C_{b,21}^w}{4q_{b,1}^w},\quad
q_{b,3}^w=\frac{C_{b,13}^w+C_{b,31}^w}{4q_{b,1}^w}.
$$
If $d_2$ is largest:
$$
q_{b,2}^w=\frac{\sqrt{\max(d_2,0)}}{2},\quad
q_{b,0}^w=\frac{C_{b,13}^w-C_{b,31}^w}{4q_{b,2}^w},\quad
q_{b,1}^w=\frac{C_{b,12}^w+C_{b,21}^w}{4q_{b,2}^w},\quad
q_{b,3}^w=\frac{C_{b,23}^w+C_{b,32}^w}{4q_{b,2}^w}.
$$
If $d_3$ is largest:
$$
q_{b,3}^w=\frac{\sqrt{\max(d_3,0)}}{2},\quad
q_{b,0}^w=\frac{C_{b,21}^w-C_{b,12}^w}{4q_{b,3}^w},\quad
q_{b,1}^w=\frac{C_{b,13}^w+C_{b,31}^w}{4q_{b,3}^w},\quad
q_{b,2}^w=\frac{C_{b,23}^w+C_{b,32}^w}{4q_{b,3}^w}.
$$
Finally normalize the result. 

### Sources:
1. [Wikipedia: Quaternion](https://en.wikipedia.org/wiki/Quaternion)
2. [Website: Visualizing Quaternions - Ben Eater & Grant Sanderson (3Blue1Brown)](https://eater.net/quaternions)
3. [Principles of GNSS, Inertial, and Multisensor Integrated Navigation Systems - Paul D. Groves - Chapter 2](../../docs/misc/Principles%20of%20GNSS,%20Inertial,%20and%20Multisensor%20Integrated%20Navigation%20Systems%20-%20Paul%20Groves.pdf)
4. [YouTube: Visualizing the 4d numbers Quaternions](https://www.youtube.com/watch?v=d4EgbgTm0Bg)
5. [Vector Algebra relations: Angles](https://en.wikipedia.org/wiki/Vector_algebra_relations#Angles)
6. [List of trigonometric identities: Half-angle formulas](https://en.wikipedia.org/wiki/List_of_trigonometric_identities#Half-angle_formulas)

## Sensor fusion: Mahony filter

### Overview

The Mahony filter is a more advanced algorithm that views sensor fusion as a closed loop control system:

* *Error*: angular velocity error derived from accelerometer readings and the true gravity vector.
* *Set point*: constant and equal to zero.
* *Controller*: linear, proportional-integral (PI).
* *Feedback*: instantaneous angular velocity and acceleration readings from the IMU.
* *Physical system*: athlete translating and rotating in space.
* *Output*: corrected angular velocity, ideally approaches zero when perfectly stationary.

In the original paper, orientation was represented using a "Special Orthogonal group (SO(3))", but in the following implementation quaternions will be used. Additionally, the convention for orientation will be changed from "world to body" (as it was done in the complementary filter) to "body to world".

### Algorithm

For each set of measurements from the IMU, apply the following steps:

1. *Transform measurements*: employing the previous orientation estimate, transform the gravity vector from the world frame to the body frame: 
   $$
   \bm{\bm{g}}^b = (\bm{q}_{b,i-1}^w)^* \otimes \bm{g}^w \otimes \bm{q}_{b,i-1}^w
   $$

2. *Compute the cost*: employ the cross product between the normalized measured specific force and the normalized gravity vector as a measure of the angular velocity error:
   $$
   \bm{e}^b = K_M (-\hat{\bm{f}}^{b} \times \hat{\bm{g}}^{b})
   $$

3. *Correct with PI feedback*: compute the gyro bias:
   $$
   \dot{\bm{b}}_{\omega,i}^b = -K_I \cdot \bm{e}^b \\
   \bm{b}_{\omega,i}^b = \bm{b}_{\omega,i-1}^b + \frac{\Delta t}{2} (\dot{\bm{b}}_{\omega,i-1}^b + \dot{\bm{b}}_{\omega,i}^b)
   $$
   Then correct the measured angular velocity with the estimated bias:
   $$
   \bm{\omega}_{corr,i}^b \leftarrow \bm{\omega}_{i}^b - \bm{b}_{\omega,i}^b + K_P \cdot \bm{e}^b
   $$
   
4. *Estimate current orientation*: integrate the quaternion kinematical equation [6]:
   $$
   \dot{\bm{q}}_{b,i}^w = \frac{1}{2} \bm{q}_{b,i-1}^w \otimes (0, \bm{\omega}_{corr,i}^b)   \\
   \bm{q}_{b,i}^w = \bm{q}_{b,i-1}^w + \frac{\Delta t}{2} (\dot{\bm{q}}_{b,i-1}^w + \dot{\bm{q}}_{b,i}^w)
   $$
   The norm of the quaternion has the property of remaining constant over time:
   $$
   \frac{d}{dt} ||\bm{q}_{b,i}^w|| = 0
   $$
   To guarantee this property numerically and avoid invalid rotations, normalize the quaternion after each integration step.

### Initial conditions

* $\bm{q}_{b,0}^w$: compute the frame of reference as discussed in the *Frames* section. Then transform the resulting matrix to a quaternion representation using the formulas discussed in *Rotations in 3D space: Quaternions - Conversion to other representations*.

* $\bm{b}_{\omega,0}$: use an offline mean of stationary gyroscope measurements from multiple captures. 

Processing should not begin until the initialization phase is over. Noisy individual samples should not corrupt the initial state.

### Interpretation of filter parameters

* *Movement error attenuation* ($K_M$): scales the error based on the device's motion. When the device is stationary, the measured acceleration should closely match the gravity vector ($K_M = 1$). When the device is in motion, dynamic acceleration affects the measurement, so the error is scaled down by a factor $K_M \in [0,1)$.

* *Proportional gain* ($K_P$): controls how aggressively the filter corrects tilt error from the accelerometer. Too high and the filter becomes jittery. Too low and the filter responds sluggishly.

* *Integral gain* ($K_I$): controls how quickly the filter estimates or learns the gyro bias. Too high and the bias estimate will chase noise. Too low and the filter will be undercorrect for real gyro bias. 

They must be tuned for optimal performance. Heuristics conventionally used such as the Ziegler-Nichols method [7] can't be used here as there is no controlled physical system to run experiments on, so trial and error tuning paired with simulation is necessary. 

### Sidenotes

The quaternion kinematical equation that maps the reference/world frame to the body frame can be obtained as follows: 
$$
\dot{\bm{q}}_{w,i}^b = 
(\dot{\bm{q}}_{b,i}^w)^{-1} = (\dot{\bm{q}}_{b,i}^w)^* =
(\frac{1}{2} \bm{q}_{b,i-1}^w \otimes (0, \bm{\omega}_i))^* =
\frac{1}{2} (0, \bm{\omega}_i)^* \otimes (\bm{q}_{b,i-1}^w)^* = 
\frac{1}{2} (0, -\bm{\omega}_i) \otimes \bm{q}_{w,i-1}^b \\
$$

$$
\boxed{\dot{\bm{q}}_{w,i}^b = -\frac{1}{2} (0, \bm{\omega}_i) \otimes \bm{q}_{w,i-1}^b}
$$

### Sources: 

1. [Nonlinear Complementary Filters on the Special Orthogonal Group](../../docs/misc/Nonlinear%20Complementary%20Filters%20on%20the%20Special%20Orthogonal%20Group%20-%20Robert%20Mahony.pdf)
2. [Complementary vs. Mahony vs. EKF: Choosing the Right Attitude Estimator for Your Drone](https://husainlokhandwala.in/2026/08/09/attitude-filter-comparison.html)
3. [IMU Mahony filter explanation](https://medium.com/@k66115704/imu-mahony-filter-explanation-1ae75bf033ab)
4. [Introducción al control de sistemas dinámicos lineales continuos - Teoría de Control - ISI - UTN FRSF](../../docs/misc/9-%20Introducción%20al%20Control%20de%20SDLC.pdf)
5. [Controlador PID - Teoría de Control - ISI - UTN FRSF](../../docs/misc/10-%20Controladores%20P+I+D.pdf)
6. [[IONLAB Lectures] Quaternion Kinematics](https://www.youtube.com/watch?v=CecyVl9iXKM)
7. [Wikipedia: Ziegler-Nichols method](https://en.wikipedia.org/wiki/Ziegier-Nichols_method)
8. [Mahony Filter - Orientation Estimation](https://itohi.com/snippets/filters/sensor-fusion-mahony/)

## Stationary detection

Previously, stationary periods were detected at the sample level as follows:
$$
\text{stationary} \iff |||\bm{a}^b_i|| - \mu_a^b| < k \sigma_a \;\wedge\; |||\bm{\omega}^b_i|| - \mu_\omega^b| < k \sigma_\omega
$$
Where $\mu_a^b$ and $\mu_\omega^b$ are the mean accelerometer and gyroscope magnitudes of stationary captures measurements, respectively, $\sigma_a$ and $\sigma_\omega$ are the corresponding standard deviations, and $k$ is a tuning parameter. 

Now this will be improved by:
* Requiring the stationary condition to hold over a window of consecutive samples. This reduces false positives.
* Use the median and median absolute deviation (MAD) instead of the mean and standard deviation. This is more robust against outliers.

If a stationary period begins at sample $j$, then for a required window of length $N$:
$$
\text{stationary} \iff |||\bm{a}^b_i|| - \text{med}(||\bm{a}^b||)| < k \text{MAD}(||\bm{a}^b||) \;\wedge\; |||\bm{\omega}^b_i|| - \text{med}(||\bm{\omega}^b||)| < k \text{MAD}(||\bm{\omega}^b||) \;\;\forall i \in [j, j+N-1]   
$$
Note that for normally distributed data $\sigma \approx 1.4826 \, \text{MAD}$.

Sources:
1. [Wikipedia: Median absolute deviation](https://en.wikipedia.org/wiki/Median_absolute_deviation)

## Setup

In [1]:
from signal_utils import (
    IMUSampleReader,
    IMUSampleWriter,
)
from signal_utils.IMUStationaryDetector import (
    InstantaneousIMUStationaryDetector,
    WindowedIMUStationaryDetector,
)
from random import seed
import matplotlib.pyplot as plt
import numpy as np
import math
from matplotlib.ticker import MultipleLocator
import os
from ahrs.filters import Mahony
from scipy import stats

SAMPLING_FREQUENCY = 30  # Hz
dt = 1 / SAMPLING_FREQUENCY
G = 9.80665  # m/s2
G_W_UNIT = np.array([0.0, 0.0, -1.0], dtype=np.float32)

sixFaceStationaryCaptures = [
    "../2-noiseReduction/output/pos-x-up-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/neg-x-up-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/pos-y-up-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/neg-y-up-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/pos-z-up-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/neg-z-up-calib-affine-sixf-tilted-a-filt-box5.csv",
]
tiltedStationaryCaptures = [
    "../2-noiseReduction/output/pos-x-pos-z-tilt-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/neg-x-pos-z-tilt-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/pos-y-pos-z-tilt-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/neg-y-pos-z-tilt-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/pos-x-neg-z-tilt-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/neg-x-neg-z-tilt-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/pos-y-neg-z-tilt-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/neg-y-neg-z-tilt-calib-affine-sixf-tilted-a-filt-box5.csv",
]
stationaryCaptures = [*sixFaceStationaryCaptures, *tiltedStationaryCaptures]

simpleRotationsCaptures = [
    "../2-noiseReduction/output/rot-x-hw-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/rot-y-hw-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/rot-z-hw-calib-affine-sixf-tilted-a-filt-box5.csv",
]

realExerciseCaptures = [
    "../2-noiseReduction/output/dips-1-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/dips-2-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/dips-3-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/pull-ups-1-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/pull-ups-2-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/pull-ups-3-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/90-deg-push-ups-1-calib-affine-sixf-tilted-a-filt-box5.csv",
    "../2-noiseReduction/output/90-deg-push-ups-2-calib-affine-sixf-tilted-a-filt-box5.csv",
]

# --------
complStationaryCaptures = [
    "./output/ast=0.750-amv=0.950/pos-x-up-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/neg-x-up-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/pos-y-up-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/neg-y-up-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/pos-z-up-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/neg-z-up-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/pos-x-pos-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/neg-x-pos-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/pos-y-pos-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/neg-y-pos-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/pos-x-neg-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/neg-x-neg-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/pos-y-neg-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/neg-y-neg-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
]
complSimpleRotationsCaptures = [
    "./output/ast=0.750-amv=0.950/rot-x-hw-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/rot-y-hw-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/rot-z-hw-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
]
complRealExerciseCaptures = [
    "./output/ast=0.750-amv=0.950/dips-1-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/dips-2-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/dips-3-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/pull-ups-1-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/pull-ups-2-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/pull-ups-3-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/90-deg-push-ups-1-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
    "./output/ast=0.750-amv=0.950/90-deg-push-ups-2-calib-affine-sixf-tilted-a-online-w-filt-box5-or-compl.csv",
]

reader = IMUSampleReader()
writer = IMUSampleWriter()

# stationaryDetector = InstantaneousIMUStationaryDetector()
# stationaryDetector.computeTolerances(stationaryCaptures)
stationaryDetector = WindowedIMUStationaryDetector(SAMPLING_FREQUENCY)
stationaryDetector.computeTolerances(stationaryCaptures)


def transformSeriesFromSensorToBodyFrame(a, w):
    aHw = a.copy()
    wHw = w.copy()

    a[:, 0] = aHw[:, 0]
    a[:, 1] = aHw[:, 2]
    a[:, 2] = -aHw[:, 1]

    w[:, 0] = wHw[:, 0]
    w[:, 1] = wHw[:, 2]
    w[:, 2] = -wHw[:, 1]

Acceleration norms:
	Median = 9.814409 m/s²
	MAD = 0.052136 (tol = 0.231892) m/s²
	Stationary interval = [9.582517, 10.046302] m/s²
Gyroscope norms:
	Median = 6.232008 deg/s
	MAD = 0.028663 (tol = 0.127488) deg/s
	Stationary interval = [6.104520, 6.359497] deg/s



## Import implemented *Quaternion* class

There's available a numpy extension called [numpy-quaternion](https://pypi.org/project/numpy-quaternion/) as well as support from scipy with [scipy.spatial.transform.Rotation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.transform.Rotation.html), but a minimal custom implementation is preferred to improve understanding and provide guidance for the firmware implementation, where no additional dependencies will be used to reduce code space. The class follows the formulas outlined above, works with inmutable objects (safer), overloads operators for convenience, and uses *float32* instead of *float64* to anticipate for precision problems in the device.

**Note**:

To compare for equality between quaternions, the underlying methods used employ different formulas:
* *np.allclose()*: $|v_{1, i} - v_{2, i}| \leq tol_a + tol_r \cdot |v_{2, i}|$
* *math.isclose()*: $|s_1 - s_2| \leq max\{tol_a, tol_r \cdot max\{|s_1|, |s_2|\}\}$

Where:
* $tol_a$ is the absolute tolerance. It's fixed, magnitude-independent, and useful for values near 0.
* $tol_r$ is the relative tolerance. It scales proportionally with the magnitude, useful for big numbers.

For simplicity, the same values of $tol_a$ and $tol_r$ are used for both methods.

Sources: 
* https://numpy.org/doc/stable/reference/generated/numpy.allclose.html
* https://docs.python.org/3/library/math.html#math.isclose

In [2]:
from signal_utils.Quaternion import Quaternion

In [3]:
def findShortestArcQuaternionFromVectors(ui: np.ndarray, uf: np.ndarray) -> Quaternion:
    uiUnit = ui / np.linalg.norm(ui)
    ufUnit = uf / np.linalg.norm(uf)
    n = np.cross(uiUnit, ufUnit)
    if np.allclose(n, np.zeros(3)):
        if np.allclose(uiUnit, ufUnit):
            nUnit = np.zeros(3, dtype=np.float32)
        else:
            # Find basis vector with least contribution in the initial vector,
            # and get it conveniently from the identity matrix
            nUnit = np.eye(3)[:, np.argmin(np.abs(uiUnit))]
    else:
        nUnit = n / np.linalg.norm(n)

    # Normalization is needed for the collinear cases
    return Quaternion(
        s=math.sqrt((1 + np.dot(uiUnit, ufUnit)) / 2),
        v=math.sqrt((1 - np.dot(uiUnit, ufUnit)) / 2) * nUnit,
    ).normalized()


def findQuaternionFromAngleAndNormalVector(
    angle: float, n: np.ndarray, inDegrees=True
) -> Quaternion:
    angleRad = angle
    if inDegrees:
        angleRad = math.radians(angle)
    return Quaternion(
        s=math.cos(angleRad / 2),
        v=math.sin(angleRad / 2) * n / np.linalg.norm(n),
    )


def findEulerAngles(s: np.ndarray, v: np.ndarray, toDegrees=True):
    eulerAngles = np.zeros((s.shape[0], 3))
    for i in range(len(s)):
        eulerAngles[i, :] = Quaternion(s[i], v[i]).toEulerAngles(toDegrees=toDegrees)
    return eulerAngles

In [4]:
# Rotation by the identity quaternion
ui = np.array([1, 0, 0], dtype=np.float32)
q = Quaternion(1, np.array([0.0, 0.0, 0.0]))
uf = q.rotate(ui)
np.testing.assert_allclose(ui, uf)

# Rotation given angle and normal vector
ui = np.array([1.0, 0.0, 0.0], dtype=np.float32)
q = findQuaternionFromAngleAndNormalVector(
    angle=90.0,
    n=np.array([0, 0, 1.0], dtype=np.float32),
    inDegrees=True,
)
uf = q.rotate(ui)
np.testing.assert_allclose(uf, np.array([0.0, 1.0, 0.0], dtype=np.float32))
assert math.isclose(np.linalg.norm(uf), 1.0, abs_tol=1e-5)

# Rotation given not collinear initial and final vectors
ui = np.array([1.0, 0.0, 0.0], dtype=np.float32)
uf = np.array([0.0, 1.0, 0.0], dtype=np.float32)
q = findShortestArcQuaternionFromVectors(ui, uf)
np.testing.assert_allclose(q.rotate(ui), uf)

# Rotation given parallel initial and final vectors
ui = np.array([1.0, 0.0, 0.0], dtype=np.float32)
uf = 2 * ui
q = findShortestArcQuaternionFromVectors(ui, uf)
uf = q.rotate(ui)
np.testing.assert_allclose(ui, uf)

# Rotation given antiparallel initial and final vectors
ui = np.array([1.0, 0.0, 0.0], dtype=np.float32)
uf = -2 * ui
q = findShortestArcQuaternionFromVectors(ui, uf)
uf = q.rotate(ui)
np.testing.assert_allclose(uf, -ui)

# Conversion from and to Euler angles
angles = np.array([30.0, 60.0, 45.0], dtype=np.float32)
np.testing.assert_allclose(
    Quaternion.fromEulerAngles(*angles).toEulerAngles(), angles, rtol=1e-5, atol=1e-5
)

# Conversion from and to Direction Cosine Matrix (DCM)
dcmBranchCases = [
    Quaternion(1.0, np.array([0.0, 0.0, 0.0])),  # d0
    Quaternion(0.0, np.array([1.0, 0.0, 0.0])),  # d1
    Quaternion(0.0, np.array([0.0, 1.0, 0.0])),  # d2
    Quaternion(0.0, np.array([0.0, 0.0, 1.0])),  # d3
]
for q in dcmBranchCases:
    assert q == Quaternion.fromDCM(q.toDCM())

## Test new stationary detection

In [6]:
# Always the same seed to be deterministic (generated numbers are pseudo-random)
rng = np.random.default_rng(12345)


class MockIMUSampleReader:
    def read(self, path):
        length = 30
        seq = np.arange(length).reshape(-1, 1)
        aSynthetic = stats.norm.rvs(
            loc=0,
            scale=0.1,
            size=(length, 3),
            random_state=rng,
        )
        aSynthetic[:, 0] += G  # type: ignore
        wSynthetic = stats.norm.rvs(
            loc=0,
            scale=0.1,
            size=(length, 3),
            random_state=rng,
        )
        return (seq, aSynthetic, wSynthetic)


# Ideally, MockIMUSampleReader and the real IMUSampleReader should implement a common interface,
# but the mock reader has the same signature and works for testing purposes
testDetector = WindowedIMUStationaryDetector(
    reader=MockIMUSampleReader(), samplingFrequency=15  # type: ignore
)  # windowSize = 3

testDetector.computeTolerances(["mockPath"])
assert testDetector.accelCenter > 0
assert testDetector.gyroCenter > 0
assert testDetector.accelTol > 0
assert testDetector.gyroTol > 0

aStationary = np.array([testDetector.accelCenter, 0, 0])
wStationary = np.array([testDetector.gyroCenter, 0, 0])
aMoving = np.array([testDetector.accelCenter + testDetector.accelTol + 1e-3, 0, 0])
wMoving = np.array([testDetector.gyroCenter + testDetector.gyroTol + 1e-3, 0, 0])

# Detects stationary sample after window has passed
for i in range(testDetector.windowSize - 1):
    assert not testDetector.isStationarySample(aStationary, wStationary)
assert testDetector.isStationarySample(aStationary, wStationary)

# Detects moving samples
assert not testDetector.isStationarySample(aMoving, wStationary)
assert not testDetector.isStationarySample(aStationary, wMoving)
assert not testDetector.isStationarySample(aMoving, wMoving)

# Resets inmediately after moving sample
for i in range(testDetector.windowSize - 1):
    assert not testDetector.isStationarySample(aStationary, wStationary)
assert not testDetector.isStationarySample(aMoving, wMoving)
assert not testDetector.isStationarySample(aStationary, wStationary)

# Doesn't detect stationary samples immediately after reset
for i in range(testDetector.windowSize - 1):
    testDetector.isStationarySample(aStationary, wStationary)
testDetector.reset()
assert not testDetector.isStationarySample(aStationary, wStationary)

Acceleration norms:
	Median = 9.795791 m/s²
	MAD = 0.067436 (tol = 0.299943) m/s²
	Stationary interval = [9.495848, 10.095733] m/s²
Gyroscope norms:
	Median = 0.146245 deg/s
	MAD = 0.042697 (tol = 0.189910) deg/s
	Stationary interval = [-0.043665, 0.336155] deg/s



## Implement Mahony filter

1. First compute offline the gyro bias from stationary captures on the sensor frame, then convert to radians and map it to the body frame, and finally feed it as the initial bias to the Mahony filter for online updates.

2. Transform the accelerometer and gyroscope series from the sensor frame to the body frame before feeding them into the Mahony filter.

In [ ]:
b0SRadPerS = np.radians(
    np.mean(
        np.concatenate([reader.read(capture)[2] for capture in stationaryCaptures]),
        axis=0,
    )
)
b0BRadPerS = np.array([b0SRadPerS[0], b0SRadPerS[2], -b0SRadPerS[1]], dtype=np.float32)
print(f"b0B(rad/s)= {b0BRadPerS}")
print(f"b0B(deg/s)= {np.degrees(b0BRadPerS)}")


Y_B = np.array([0.0, 1.0, 0.0], dtype=np.float32)
Z_B = np.array([0.0, 0.0, 1.0], dtype=np.float32)
HORIZONTAL_TOL = 0.01


def buildReferenceFrame(fBMean: np.ndarray):
    zWBUnit = fBMean / np.linalg.norm(fBMean)

    yWB = Y_B - np.dot(Y_B, zWBUnit) * zWBUnit
    if np.linalg.norm(yWB) < HORIZONTAL_TOL:
        yWB = Z_B - np.dot(Z_B, zWBUnit) * zWBUnit
    yWBUnit = yWB / np.linalg.norm(yWB)

    xWB = np.cross(yWBUnit, zWBUnit)
    xWBUnit = xWB / np.linalg.norm(xWB)

    return np.array([xWBUnit, yWBUnit, zWBUnit], dtype=np.float32)

In [ ]:
# Matrix is orthogonal (i.e. columns form an orthonormal base)
fBMeans = [
    np.array([0.0, 0.0, G], dtype=np.float32),  # Standing upright
    np.array(
        [0.0, -G / math.sqrt(2), G / math.sqrt(2)], dtype=np.float32
    ),  # Leaning forwards by 45 degrees
    np.array([0.0, G, 0.0], dtype=np.float32),  # Horizontal, resting on the back
    np.array([G, 0.0, 0.0], dtype=np.float32),  # Sideways, to the left
]
for fBMean in fBMeans:
    Cbw = buildReferenceFrame(fBMean)
    np.testing.assert_allclose(Cbw.T, np.linalg.inv(Cbw))

In [ ]:
INIT_TIME_SECONDS = 1
INIT_SAMPLE_COUNT = math.ceil(INIT_TIME_SECONDS * SAMPLING_FREQUENCY)


def mahonyFilter(fB, wBDeg, dt, km, kp, ki):
    wBRad = np.radians(wBDeg)

    initFBMean = np.mean(fB[:INIT_SAMPLE_COUNT], axis=0)
    initWBRadMean = np.mean(wBRad[:INIT_SAMPLE_COUNT], axis=0)
    processedFB = fB[INIT_SAMPLE_COUNT - 1 :]
    processedWBRadPerS = wBRad[INIT_SAMPLE_COUNT - 1 :]

    procSamplesCount = len(processedWBRadPerS)
    s = np.zeros(procSamplesCount, dtype=np.float32)
    v = np.zeros((procSamplesCount, 3), dtype=np.float32)
    bBRadPerS = np.zeros((procSamplesCount, 3), dtype=np.float32)
    eB = np.zeros((procSamplesCount, 3), dtype=np.float32)
    correctedWBRadPerS = np.zeros((procSamplesCount, 3), dtype=np.float32)

    processedFB[0] = initFBMean
    processedWBRadPerS[0] = initWBRadMean - b0BRadPerS
    correctedWBRadPerS[0] = processedWBRadPerS[0]
    # q0 = findShortestArcQuaternionFromVectors(-initFBMean, G_W_UNIT)
    q0 = Quaternion.fromDCM(buildReferenceFrame(initFBMean))
    s[0], v[0] = q0.s, q0.v
    prevQ = q0
    prevQRate = 1 / 2 * q0 * Quaternion(0.0, correctedWBRadPerS[0])
    bBRadPerS[0] = b0BRadPerS
    prevBRate = np.zeros(3, dtype=np.float32)

    for i in range(1, len(s)):
        gBUnit = prevQ.conjugate().rotate(G_W_UNIT)

        eB[i] = np.cross(-processedFB[i] / np.linalg.norm(processedFB[i]), gBUnit)
        if not stationaryDetector.isStationarySample(
            processedFB[i, :], np.degrees(processedWBRadPerS[i, :])
        ):
            eB[i] *= km

        bRate = -ki * eB[i]
        bBRadPerS[i] = bBRadPerS[i - 1] + dt / 2 * (prevBRate + bRate)
        # bBRadPerS[i] = bBRadPerS[i - 1] + dt * bRate
        prevBRate = bRate

        processedWBRadPerS[i] -= bBRadPerS[i]
        correctedWBRadPerS[i] = processedWBRadPerS[i] + kp * eB[i]

        qRate = 1 / 2 * prevQ * Quaternion(0.0, correctedWBRadPerS[i])
        q = (prevQ + dt / 2 * (prevQRate + qRate)).normalized()
        # q = (prevQ + dt * qRate).normalized()
        s[i], v[i] = q.s, q.v
        prevQ = q
        prevQRate = qRate

    return (
        processedFB,
        np.degrees(processedWBRadPerS),
        s,
        v,
        np.degrees(correctedWBRadPerS),
        bBRadPerS,
        eB,
    )

## Find "optimal" parameter values for the filter

Perform a grid search with different values for KM, KP, and KI to later analyze gravity removal effectiveness. Possible range of values:
* $K_M$: $[0, 1)$
* $K_P$: $(0, 10]$
* $K_I$: $[0, 0.1]$

Be careful with the grid size, this is **S-L-O-W**!

In [ ]:
KMRange = np.arange(0.25, 1.0, 0.25)
KPRange = np.arange(1.0, 5.5, 1.0)
KIRange = np.arange(0, 0.125, 0.025)
captures = [
    *stationaryCaptures,
    *simpleRotationsCaptures,
    *realExerciseCaptures,
]
print(f"Grid:\tKM: {KMRange}, \n\tKP: {KPRange}, \n\tKI: {KIRange}")

for km in KMRange:
    for kp in KPRange:
        for ki in KIRange:
            for capture in captures:
                seq, fB, wB = reader.read(capture)
                transformSeriesFromSensorToBodyFrame(fB, wB)
                fB, wB, s, v, _, _, _ = mahonyFilter(fB, wB, dt, km, kp, ki)
                ownImplMahonyAngles = findEulerAngles(s, v, toDegrees=True)
                oldFilename = f"{capture.split('/')[-1].split('.')[0]}"
                directory = f"./output/mahony/km={km:.3f}-kp={kp:.3f}-ki={ki:.3f}/"
                os.makedirs(directory, exist_ok=True)
                writer.write(
                    outputFile=f"{directory}{oldFilename}-or-mahony.csv",
                    seq=seq[INIT_SAMPLE_COUNT - 1 :],
                    a=fB,
                    w=wB,
                    angle=ownImplMahonyAngles,
                    q=np.hstack((s[:, np.newaxis], v)),
                )

## Plot results for a given set of parameters

Compare against:
* A reference implementation from the AHRS library to address overall correctness: [documentation](https://ahrs.readthedocs.io/en/latest/filters/mahony.html#module-ahrs.filters.mahony), [code](https://github.com/Mayitzin/ahrs/blob/master/ahrs/filters/mahony.py). To do it, temporarily set $K_M = 1$ and use the same initial orientation and bias, as well as trimmed accelerometer and gyroscope data, to make the comparison more accurate.
* My own complementary filter: to address whether the results are better than my previous implementation.

In [ ]:
SMALL_ANGLE_RANGE_THRESHOLD = 3


def plotOrientationEulerAngles(
    seq,
    ownImplMahonyAngles,
    ahrsMahonyAngles,
    ownImplComplAngles,
    stationaryIntervals,
):
    def configGraph(upperBound, lowerBound, axis, title):
        if lowerBound < -180:
            axis.axhline(-180, color="black", linestyle="-", linewidth=2.0, zorder=0)
        if lowerBound < -90:
            axis.axhline(-90, color="black", linestyle="-", linewidth=2.0, zorder=0)
        axis.axhline(0, color="black", linestyle="-", linewidth=2.0, zorder=0)
        if upperBound > 90:
            axis.axhline(90, color="black", linestyle="-", linewidth=2.0, zorder=0)
        if upperBound > 180:
            axis.axhline(180, color="black", linestyle="-", linewidth=2.0, zorder=0)
        axis.set_ylabel("Angle (degrees)")
        if upperBound - lowerBound > SMALL_ANGLE_RANGE_THRESHOLD:
            axis.yaxis.set_major_locator(MultipleLocator(15))

        axis.grid(True, which="both", linestyle="-", alpha=0.75)
        axis.set_xlabel("Sequence number")
        axis.set_title(title)

        for lower, upper in stationaryIntervals:
            axis.axvspan(lower, upper, color="tab:gray", alpha=0.25, zorder=0)

        axis.legend()

    lowerBoundRoll = min(
        ownImplMahonyAngles[:, 0].min(),
        ahrsMahonyAngles[:, 0].min(),
        ownImplComplAngles[:, 0].min(),
    )
    upperBoundRoll = max(
        ownImplMahonyAngles[:, 0].max(),
        ahrsMahonyAngles[:, 0].max(),
        ownImplComplAngles[:, 0].max(),
    )
    lowerBoundPitch = min(
        ownImplMahonyAngles[:, 1].min(),
        ahrsMahonyAngles[:, 1].min(),
        ownImplComplAngles[:, 1].min(),
    )
    upperBoundPitch = max(
        ownImplMahonyAngles[:, 1].max(),
        ahrsMahonyAngles[:, 1].max(),
        ownImplComplAngles[:, 1].max(),
    )
    lowerBoundYaw = min(
        ahrsMahonyAngles[:, 2].min(),
        ownImplComplAngles[:, 2].min(),
        ownImplMahonyAngles[:, 2].min(),
    )
    upperBoundYaw = max(
        ahrsMahonyAngles[:, 2].max(),
        ownImplComplAngles[:, 2].max(),
        ownImplMahonyAngles[:, 2].max(),
    )

    paddedOwnImplMahonyAngles = np.concatenate(
        [np.nan * np.ones((INIT_SAMPLE_COUNT - 1, 3)), ownImplMahonyAngles]
    )
    paddedAhrsMahonyAngles = np.concatenate(
        [np.nan * np.ones((INIT_SAMPLE_COUNT - 1, 3)), ahrsMahonyAngles]
    )

    fig, axes = plt.subplots(3, 1, figsize=(16, 16))

    axes[0].plot(
        seq,
        paddedOwnImplMahonyAngles[:, 0],
        label="Mahony filter (own implementation)",
        color="blue",
    )
    axes[0].plot(
        seq,
        paddedAhrsMahonyAngles[:, 0],
        label="Mahony filter (AHRS library)",
        color="green",
    )
    axes[0].plot(
        seq, ownImplComplAngles[:, 0], label="Complementary filter", color="red"
    )
    configGraph(upperBoundRoll, lowerBoundRoll, axes[0], "Roll angle")

    axes[1].plot(
        seq,
        paddedOwnImplMahonyAngles[:, 1],
        label="Mahony filter (own implementation)",
        color="blue",
    )
    axes[1].plot(
        seq,
        paddedAhrsMahonyAngles[:, 1],
        label="Mahony filter (AHRS library)",
        color="green",
    )
    axes[1].plot(
        seq, ownImplComplAngles[:, 1], label="Complementary filter", color="red"
    )
    configGraph(upperBoundPitch, lowerBoundPitch, axes[1], "Pitch angle")

    axes[2].plot(
        seq,
        paddedOwnImplMahonyAngles[:, 2],
        label="Mahony filter (own implementation)",
        color="blue",
    )
    axes[2].plot(
        seq,
        paddedAhrsMahonyAngles[:, 2],
        label="Mahony filter (AHRS library)",
        color="green",
    )
    axes[2].plot(
        seq, ownImplComplAngles[:, 2], label="Complementary filter", color="red"
    )
    configGraph(upperBoundYaw, lowerBoundYaw, axes[2], "Yaw angle")

    fig.suptitle(f"Orientation comparison: Complementary vs Mahony\n")
    plt.tight_layout()
    plt.show()


def plotOrientationQuaternions(seq, s, v):
    fig, axis = plt.subplots(1, 1, figsize=(12, 6))

    axis.plot(seq[INIT_SAMPLE_COUNT - 1 :], s[:], label="q0", color="blue")
    axis.plot(seq[INIT_SAMPLE_COUNT - 1 :], v[:, 0], label="q1", color="green")
    axis.plot(seq[INIT_SAMPLE_COUNT - 1 :], v[:, 1], label="q2", color="red")
    axis.plot(seq[INIT_SAMPLE_COUNT - 1 :], v[:, 2], label="q3", color="purple")
    axis.plot(
        seq[INIT_SAMPLE_COUNT - 1 :],
        np.sqrt(s**2 + v[:, 0] ** 2 + v[:, 1] ** 2 + v[:, 2] ** 2),
        label="Norm",
        color="orange",
    )
    axis.set_ylabel("Quaternion component value")
    axis.set_xlabel("Sequence number")
    axis.legend()
    axis.grid(True, which="both", linestyle="-", alpha=0.75)
    fig.suptitle("Quaternion components and norm")
    plt.tight_layout()
    plt.show()


def plotErrors(seq, e):
    fig, axis = plt.subplots(1, 1, figsize=(12, 6))

    axis.plot(seq[INIT_SAMPLE_COUNT - 1 :], e[:, 0], label="Roll error", color="blue")
    axis.plot(seq[INIT_SAMPLE_COUNT - 1 :], e[:, 1], label="Pitch error", color="green")
    axis.plot(seq[INIT_SAMPLE_COUNT - 1 :], e[:, 2], label="Yaw error", color="red")
    axis.grid(True, which="both", linestyle="-", alpha=0.75)
    axis.set_ylabel("Error")
    axis.set_xlabel("Sequence number")
    axis.legend()
    fig.suptitle(f"Orientation errors")
    plt.tight_layout()
    plt.show()


def plotBias(seq, b):
    fig, axis = plt.subplots(1, 1, figsize=(12, 6))

    axis.plot(seq[INIT_SAMPLE_COUNT - 1 :], b[:, 0], label="Roll bias", color="blue")
    axis.plot(seq[INIT_SAMPLE_COUNT - 1 :], b[:, 1], label="Pitch bias", color="green")
    axis.plot(seq[INIT_SAMPLE_COUNT - 1 :], b[:, 2], label="Yaw bias", color="red")
    axis.grid(True, which="both", linestyle="-", alpha=0.75)
    axis.set_ylabel("Bias (deg/s)")
    axis.set_xlabel("Sequence number")
    axis.legend()
    fig.suptitle(f"Orientation biases")
    plt.tight_layout()
    plt.show()

In [ ]:
# Change indexes to visualize different captures. Make sure to match the Mahony and complementary filter captures

# previousStepCapturePath = stationaryCaptures[2]
# previousStepCapturePath = simpleRotationsCaptures[1]
previousStepCapturePath = realExerciseCaptures[4]

# complOutputPath = complStationaryCaptures[2]
# complOutputPath = complSimpleRotationsCaptures[1]
complOutputPath = complRealExerciseCaptures[4]

seq, fB, wB = reader.read(previousStepCapturePath)
transformSeriesFromSensorToBodyFrame(fB, wB)
stationaryIntervals = stationaryDetector.findStationaryIntervals(seq, fB, wB)
km = 0.25
kp = 1.0
ki = 0.05
fB, wBBiasFree, s, v, wCorr, b, e = mahonyFilter(fB, wB, dt, km, kp, ki)
ownImplMahonyAngles = findEulerAngles(s, v, toDegrees=True)

ahrsMahonyFilter = Mahony(
    acc=fB.astype(np.float64),
    gyr=np.radians(wB[INIT_SAMPLE_COUNT - 1 :]).astype(np.float64),
    frequency=SAMPLING_FREQUENCY,
    kp=kp,
    ki=ki,
    q0=np.array([s[0], *v[0]]),
    b0=b[0].astype(np.float64),
)
ahrsMahonyQuaternions = ahrsMahonyFilter.Q
ahrsMahonyAngles = findEulerAngles(
    ahrsMahonyQuaternions[:, 0],
    ahrsMahonyQuaternions[:, 1:],
    toDegrees=True,
)

_, _, _, complAngles = reader.read(complOutputPath)

print(
    f"Mahony filter applied to: \t{previousStepCapturePath} \nComplementary filter from: \t{complOutputPath}"
)
plotOrientationEulerAngles(
    seq, ownImplMahonyAngles, ahrsMahonyAngles, complAngles, stationaryIntervals
)

In [ ]:
plotOrientationQuaternions(seq, s, v)

In [ ]:
plotErrors(seq, e)

In [ ]:
plotBias(seq, np.degrees(b))